In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import h5py
import numpy as np
import json
import gc
import sys
import os
import copy
from typing import List, Dict, Optional, Tuple
from sklearn.model_selection import train_test_split
import pandas as pd 
from sklearn.metrics import confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm

In [2]:
# Set style for better visualizations
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print("✅ Libraries imported successfully")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

✅ Libraries imported successfully
PyTorch version: 2.5.1+cu121
CUDA available: True
CUDA device: NVIDIA GeForce RTX 3050 Laptop GPU


In [3]:
FILE_PATH = "/home/lipplopp/research/AMC_Repository/dataset/GOLD_XYZ_OSC.0001_1024.hdf5"
JSON_PATH = '/home/lipplopp/research/AMC_Repository/dataset/classes-fixed.json' 
BATCH_SIZE = 64  # Adjust based on your GPU memory
NUM_WORKERS = 4  # Adjust based on your CPU cores
patience = 10
TARGET_MODULATIONS = [
                      #'OOK',
                      # '4ASK',
                      '8ASK',
                      'BPSK', 
                      #'QPSK',
                      '8PSK', 
                      '16QAM',
                      '64QAM', 
                      #'OQPSK'
                     ]
NUM_CLASSES = len(TARGET_MODULATIONS)
NUM_EPOCHS = 100
SUBSAMPLE_TRAIN_RATIO = 0.2  # Use 20% of training data (set to 1.0 for full data)
TRAIN_SIZE = 0.7
VALID_SIZE = 0.2
TEST_SIZE = 0.1
SPLIT_SEED = 48
NORM_SEED = 49
CHUNK_SIZE = 10000 

# --- Parameter Model (Baru) ---
PATCH_SIZE = 4       # Ukuran patch (misal: 4x4)
D_MODEL = 128        # Dimensi embedding
N_HEAD = 8           # Jumlah attention heads
N_LAYERS = 2         # Jumlah lapisan encoder
FFN_HIDDEN = D_MODEL * 4 # Ukuran hidden layer di FFN
DROP_PROB = 0.1      # Dropout probability
LEARNING_RATE = 1e-4

In [4]:
# --- Path Setup ---
# Asumsikan notebook ini ada di /home/lipplopp/research/AMC_Repository/transformer/
# Kita tambahkan parent directory-nya ('transformer') ke path agar bisa mengimpor 'models'
notebook_dir = os.getcwd()
project_root = os.path.dirname(notebook_dir) # Seharusnya /home/lipplopp/research/AMC_Repository/
# Modul model ada di 'transformer/models'
model_path = os.path.join(notebook_dir, 'models') 
# Kita tambahkan directory 'transformer' ke path
if notebook_dir not in sys.path:
    sys.path.append(notebook_dir)
print(f"Menambahkan {notebook_dir} ke sys.path")

try:
    from models.amc_transformer import AMCTransformer
    print("✅ Modul AMCTransformer berhasil diimpor.")
except ImportError as e:
    print(f"❌ GAGAL mengimpor model: {e}")
    print("Pastikan notebook ini dijalankan dari dalam direktori 'transformer'!")

# --- Setup Device ---
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"🔧 Konfigurasi dimuat")
print(f"🎯 Target Modulasi: {TARGET_MODULATIONS} ({NUM_CLASSES} kelas)")
print(f"💾 Batch size: {BATCH_SIZE}")
print(f"🖥️ Device: {device}")

Menambahkan /home/lipplopp/research/AMC_Repository/transformer ke sys.path
✅ Modul AMCTransformer berhasil diimpor.
🔧 Konfigurasi dimuat
🎯 Target Modulasi: ['8ASK', 'BPSK', '8PSK', '16QAM', '64QAM'] (5 kelas)
💾 Batch size: 64
🖥️ Device: cuda


In [5]:
def load_dataset_metadata(file_path: str, json_path: str) -> Tuple[np.ndarray, np.ndarray, List[str], int]:
    """
    Load only metadata and labels from HDF5 file (memory efficient).
    
    Returns:
        (Y_strings, Z_data, available_modulations, total_samples)
    """
    print("📂 Loading dataset metadata (memory efficient)...")
    
    with h5py.File(file_path, 'r') as hdf5_file:
        # Get dataset shape without loading data
        total_samples = hdf5_file['X'].shape[0]
        signal_length = hdf5_file['X'].shape[1]
        num_channels = hdf5_file['X'].shape[2]
        
        print(f"📊 Dataset shape: ({total_samples:,} × {signal_length} × {num_channels})")
        
        # Load only labels (much smaller than signal data)
        Y_int = np.argmax(hdf5_file['Y'][:], axis=1)
        Z_data = hdf5_file['Z'][:, 0]
        
        # Load modulation classes
        with open(json_path, 'r') as f:
            modulation_classes = json.load(f)
        
        # Convert integer labels to string labels
        Y_strings = np.array([modulation_classes[i] for i in Y_int])
        
        # Get available modulations
        available_modulations = list(np.unique(Y_strings))
        
        print(f"✅ Metadata loaded: {total_samples:,} samples")
        print(f"📡 Available modulations: {len(available_modulations)}")
        print(f"📊 SNR range: {np.min(Z_data):.1f} to {np.max(Z_data):.1f} dB")
        
        # Memory usage estimate
        data_size_gb = (total_samples * signal_length * num_channels * 4) / (1024**3)
        print(f"💾 Full dataset size: ~{data_size_gb:.2f} GB")
        
    return Y_strings, Z_data, available_modulations, total_samples

In [6]:
def stratified_dataset_split_memory_efficient(
    modulations: np.ndarray, 
    snrs: np.ndarray, 
    target_modulations: List[str],
    train_size: float = 0.7, 
    valid_size: float = 0.2, 
    test_size: float = 0.1,
    seed: int = 48, 
    subsample_train_ratio: Optional[float] = None) -> Tuple[Dict, Dict]:
    """
    Memory-efficient stratified split (doesn't require loading X data).
    
    Args:
        modulations: Modulation labels (Y)
        snrs: SNR labels (Z)
        target_modulations: Modulations to include
        train_size, valid_size, test_size: Split proportions
        seed: Random seed
        subsample_train_ratio: Optional subsampling of training data
    
    Returns:
        (splits_dict, label_map)
    """
    
    # Input validation
    if not np.isclose(train_size + valid_size + test_size, 1.0, atol=1e-6):
        raise ValueError(f"Split sizes must sum to 1.0")
    
    if len(target_modulations) == 0:
        raise ValueError("target_modulations cannot be empty")
    
    print("🔄 Performing stratified split...")
    
    # Filter to include only target modulations
    target_mask = np.isin(modulations, target_modulations)
    target_indices = np.where(target_mask)[0]
    
    if len(target_indices) == 0:
        raise ValueError("No samples found for target modulations")
    
    print(f"📊 Found {len(target_indices):,} samples for target modulations")
    
    filtered_modulations = modulations[target_indices]
    filtered_snrs = snrs[target_indices]
    
    # Create stratification key
    stratify_key = [f"{mod}_{snr}" for mod, snr in zip(filtered_modulations, filtered_snrs)]
    
    # Check sample distribution
    unique_keys, key_counts = np.unique(stratify_key, return_counts=True)
    min_samples_per_key = np.min(key_counts)
    
    if min_samples_per_key < 2:
        warnings.warn(f"⚠️ Some modulation-SNR combinations have only {min_samples_per_key} samples.")
    
    # Set random seed
    np.random.seed(seed)
    
    # First split: separate test set
    train_val_indices, test_indices = train_test_split(
        target_indices, 
        test_size=test_size, 
        random_state=seed, 
        stratify=stratify_key
    )
    
    # Second split: separate train and validation
    remaining_modulations = modulations[train_val_indices]
    remaining_snrs = snrs[train_val_indices]
    remaining_stratify_key = [f"{mod}_{snr}" for mod, snr in zip(remaining_modulations, remaining_snrs)]
    
    relative_valid_size = valid_size / (1 - test_size)
    
    train_indices, valid_indices = train_test_split(
        train_val_indices,
        test_size=relative_valid_size,
        random_state=seed,
        stratify=remaining_stratify_key
    )
    
    # Apply subsampling if requested
    if subsample_train_ratio is not None and subsample_train_ratio < 1.0:
        n_keep = int(len(train_indices) * subsample_train_ratio)
        if n_keep == 0:
            raise ValueError("Subsampling ratio too small")
        
        np.random.seed(seed)
        train_indices = np.random.choice(train_indices, n_keep, replace=False)
        print(f"📉 Subsampled training data to {n_keep:,} samples ({subsample_train_ratio:.1%})")
    
    # Create label mapping
    label_map = {name: i for i, name in enumerate(target_modulations)}
    
    # Print statistics
    print(f"\n✅ Dataset split completed:")
    print(f"  📚 Train: {len(train_indices):,} samples")
    print(f"  🔍 Valid: {len(valid_indices):,} samples") 
    print(f"  🧪 Test: {len(test_indices):,} samples")
    
    splits = {
        'train': train_indices,
        'valid': valid_indices,
        'test': test_indices
    }
    
    return splits, label_map

print("✅ Memory-efficient functions defined")

✅ Memory-efficient functions defined


In [7]:
class SingleStreamImageDataset(Dataset):
    """
    Versi modifikasi dari dataset Anda.
    - Mengubah H, W menjadi 32, 64.
    - Menghapus perhitungan Amplitudo/Fase.
    - Menghasilkan satu 'gambar' I/Q gabungan ber-shape [1, 32, 64].
    """
    
    def __init__(self, 
                 file_path: str, 
                 json_path: str, 
                 target_modulations: List[str], 
                 mode: str, 
                 indices: np.ndarray, 
                 label_map: Dict[str, int], 
                 normalization_stats: Optional[Dict] = None, 
                 seed: int = 49):
        super(SingleStreamImageDataset, self).__init__()

        # Validasi
        if mode not in ['train', 'valid', 'test']:
            raise ValueError(f"mode must be 'train', 'valid', or 'test'")
        if len(indices) == 0:
            raise ValueError("indices cannot be empty")
            
        self.file_path = file_path
        self.json_path = json_path
        self.target_modulations = target_modulations
        self.mode = mode
        self.indices = np.array(indices, dtype=int)
        self.label_map = label_map
        self.seed = seed

        self.hdf5_file = h5py.File(self.file_path, 'r')
        self.X_h5 = self.hdf5_file['X']
        
        self.Y_int = np.argmax(self.hdf5_file['Y'][:], axis=1)
        self.Z = self.hdf5_file['Z'][:, 0]

        with open(self.json_path, 'r') as f:
            self.modulation_classes = json.load(f)

        self.Y_strings = np.array([self.modulation_classes[i] for i in self.Y_int])

        signal_length = self.X_h5.shape[1]
        if signal_length != 1024:
            raise ValueError(f"Expected signal length 1024, got {signal_length}")
        
        # --- PERUBAHAN DI SINI ---
        # 1024 (I) + 1024 (Q) = 2048. 32 * 64 = 2048.
        self.H, self.W = 32, 64

        if mode == 'train':
            if normalization_stats is None:
                print(f"📊 Menghitung statistik normalisasi untuk mode {mode}...")
                self.norm_stats = self._calculate_normalization_stats()
                print(f"✅ Statistik dihitung")
            else:
                self.norm_stats = normalization_stats
        else:
            if normalization_stats is None:
                raise ValueError(f"normalization_stats wajib ada untuk mode '{mode}'")
            self.norm_stats = normalization_stats

        print(f"✅ {mode.capitalize()} dataset: {len(self.indices):,} sampel (mode memori-efisien)")

    def _calculate_normalization_stats(self) -> Dict[str, float]:
        """Menghitung statistik normalisasi (hanya I/Q) menggunakan chunked processing."""
        num_samples = min(5000, len(self.indices))
        np.random.seed(self.seed)
        sample_indices = np.random.choice(self.indices, num_samples, replace=False)
        sorted_indices = np.sort(sample_indices)
        
        chunk_size = min(500, num_samples)
        i_vals, q_vals = [], []
        
        print(f"  Memproses {num_samples} sampel dalam {len(sorted_indices)//chunk_size + 1} chunk...")
        for i in range(0, len(sorted_indices), chunk_size):
            chunk_indices = sorted_indices[i:i+chunk_size]
            chunk_data = self.X_h5[chunk_indices, ...]
            chunk_tensor = torch.from_numpy(chunk_data).float()
            
            i_vals.append(chunk_tensor[:, :, 0].flatten())
            q_vals.append(chunk_tensor[:, :, 1].flatten())
            del chunk_data, chunk_tensor
        
        i_all = torch.cat(i_vals)
        q_all = torch.cat(q_vals)
        
        stats = {
            'i_mean': i_all.mean().item(),
            'i_std': max(i_all.std().item(), 1e-8),
            'q_mean': q_all.mean().item(), 
            'q_std': max(q_all.std().item(), 1e-8)
        }
        
        del i_vals, q_vals, i_all, q_all
        gc.collect()
        return stats

    def __len__(self) -> int:
        return len(self.indices)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, int, float]:
        """Mendapatkan satu sampel sebagai 'gambar' I/Q gabungan."""
        if idx >= len(self.indices):
            raise IndexError(f"Index {idx} di luar jangkauan")
            
        true_index = int(self.indices[idx])
        
        x_raw = self.X_h5[true_index]
        y_string = self.Y_strings[true_index]
        z = float(self.Z[true_index])
        
        y = self.label_map[y_string]

        iq_sequence = torch.from_numpy(x_raw.copy()).float()
        iq_sequence[:, 0] = (iq_sequence[:, 0] - self.norm_stats['i_mean']) / self.norm_stats['i_std']
        iq_sequence[:, 1] = (iq_sequence[:, 1] - self.norm_stats['q_mean']) / self.norm_stats['q_std']

        # --- INTI PERUBAHAN ---
        # 1. Ambil sinyal I dan Q yang sudah dinormalisasi
        i_signal = iq_sequence[:, 0]
        q_signal = iq_sequence[:, 1]
        
        # 2. Gabungkan keduanya
        iq_concat = torch.cat((i_signal, q_signal), dim=0) # Shape: [2048]
        
        # 3. Ubah bentuknya menjadi "gambar" 2D (Channel, Height, Width)
        iq_image = iq_concat.view(1, self.H, self.W) # Shape: [1, 32, 64]
        # --- AKHIR PERUBAHAN ---
        
        return iq_image, y, z

    def get_normalization_stats(self) -> Dict[str, float]:
        return self.norm_stats.copy()

    def close(self):
        if hasattr(self, 'hdf5_file') and self.hdf5_file is not None:
            try:
                self.hdf5_file.close()
                print(f"🔒 {self.mode.capitalize()} dataset: File HDF5 ditutup")
            except: pass
            finally: self.hdf5_file = None

    def __del__(self):
        self.close()

print("✅ Kelas SingleStreamImageDataset didefinisikan.")

✅ Kelas SingleStreamImageDataset didefinisikan.


In [8]:
def split_data(file_path, json_path, target_mods, train_ratio, valid_ratio, test_ratio, seed):
    print("Membagi data...")
    if not np.isclose(train_ratio + valid_ratio + test_ratio, 1.0):
        raise ValueError("Rasio split harus berjumlah 1.0")
        
    # 1. Buat label_map dari target
    label_map = {mod: i for i, mod in enumerate(target_mods)}
    
    # 2. Muat semua label dan SNR
    with h5py.File(file_path, 'r') as f:
        Y_int = np.argmax(f['Y'][:], axis=1)
        Z = f['Z'][:, 0]
        
    with open(json_path, 'r') as file:
        all_mod_classes = json.load(file)
    
    Y_strings = np.array([all_mod_classes[i] for i in Y_int])
    
    train_indices, valid_indices, test_indices = [], [], []
    
    # 3. Stratifikasi berdasarkan Modulasi DAN SNR
    for mod in target_mods:
        for snr in np.unique(Z):
            # Dapatkan semua indeks untuk pasangan (mod, snr) ini
            idx = np.where((Y_strings == mod) & (Z == snr))[0]
            if len(idx) == 0:
                continue
            
            # Split pertama: pisahkan data Uji
            idx_train_val, idx_test = train_test_split(
                idx, test_size=test_ratio, random_state=seed, shuffle=True
            )
            
            # Hitung ulang rasio validasi untuk split kedua
            relative_valid_ratio = valid_ratio / (train_ratio + valid_ratio)
            
            # Split kedua: pisahkan data Latih dan Validasi
            if len(idx_train_val) > 1:
                idx_train, idx_valid = train_test_split(
                    idx_train_val, test_size=relative_valid_ratio, random_state=seed, shuffle=True
                )
            else: # Jika hanya tersisa 1 sampel, masukkan ke training
                idx_train, idx_valid = idx_train_val, []

            train_indices.extend(idx_train)
            valid_indices.extend(idx_valid)
            test_indices.extend(idx_test)
            
    # 4. Acak hasil akhir
    np.random.seed(seed)
    np.random.shuffle(train_indices)
    np.random.shuffle(valid_indices)
    np.random.shuffle(test_indices)
    
    print(f"Data terbagi: {len(train_indices)} Latih, {len(valid_indices)} Validasi, {len(test_indices)} Uji")
    return np.array(train_indices), np.array(valid_indices), np.array(test_indices), label_map

print("✅ Fungsi split_data didefinisikan.")

✅ Fungsi split_data didefinisikan.


In [9]:
# Jalankan pemisahan data
train_indices, valid_indices, test_indices, label_map = split_data(
    FILE_PATH, JSON_PATH, TARGET_MODULATIONS, TRAIN_SIZE, VALID_SIZE, TEST_SIZE, SPLIT_SEED
)

# 1. Buat dataset Latih dan dapatkan statistik normalisasi
train_dataset = SingleStreamImageDataset(
    file_path=FILE_PATH, json_path=JSON_PATH, target_modulations=TARGET_MODULATIONS,
    mode='train', indices=train_indices, label_map=label_map, seed=NORM_SEED
)
norm_stats = train_dataset.get_normalization_stats()
print(f"Statistik Normalisasi: {norm_stats}")

# 2. Buat dataset Validasi (menggunakan statistik dari Latih)
valid_dataset = SingleStreamImageDataset(
    file_path=FILE_PATH, json_path=JSON_PATH, target_modulations=TARGET_MODULATIONS,
    mode='valid', indices=valid_indices, label_map=label_map, normalization_stats=norm_stats
)

# 3. Buat dataset Uji (menggunakan statistik dari Latih)
test_dataset = SingleStreamImageDataset(
    file_path=FILE_PATH, json_path=JSON_PATH, target_modulations=TARGET_MODULATIONS,
    mode='test', indices=test_indices, label_map=label_map, normalization_stats=norm_stats
)

# 4. Buat DataLoaders
train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, 
    num_workers=NUM_WORKERS, pin_memory=True
)
valid_loader = DataLoader(
    valid_dataset, batch_size=BATCH_SIZE, shuffle=False, 
    num_workers=NUM_WORKERS, pin_memory=True
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False, 
    num_workers=NUM_WORKERS, pin_memory=True
)

print("\n✅ DataLoaders siap.")

Membagi data...
Data terbagi: 372580 Latih, 106600 Validasi, 53300 Uji
📊 Menghitung statistik normalisasi untuk mode train...
  Memproses 5000 sampel dalam 11 chunk...
✅ Statistik dihitung
✅ Train dataset: 372,580 sampel (mode memori-efisien)
Statistik Normalisasi: {'i_mean': 0.0025397716090083122, 'i_std': 0.8000569343566895, 'q_mean': 0.000767325225751847, 'q_std': 0.8086669445037842}
✅ Valid dataset: 106,600 sampel (mode memori-efisien)
✅ Test dataset: 53,300 sampel (mode memori-efisien)

✅ DataLoaders siap.


In [10]:
# Mari kita periksa satu batch
print("Melakukan sanity check pada DataLoader...")
try:
    images, labels, snrs = next(iter(train_loader))
    print(f"Shape batch gambar: {images.shape}")
    print(f"Shape batch label:  {labels.shape}")
    print(f"Contoh SNR:         {snrs[:5].tolist()}")
    
    # Harapannya [BATCH_SIZE, 1, 32, 64]
    assert images.shape == (BATCH_SIZE, 1, 32, 64) or images.shape[1:] == (1, 32, 64)
    print("✅ Sanity check berhasil!")
except Exception as e:
    print(f"❌ Sanity check GAGAL: {e}")

Melakukan sanity check pada DataLoader...
Shape batch gambar: torch.Size([64, 1, 32, 64])
Shape batch label:  torch.Size([64])
Contoh SNR:         [-12.0, 18.0, 14.0, -6.0, 26.0]
✅ Sanity check berhasil!


In [12]:
print("Menginisialisasi model...")

# Tentukan parameter input untuk model
# (Berdasarkan `test_model.py` dan `patch_embedding.py`)
model_params = {
    'in_channels': 1,
    'img_size_h': 32,    # Tambahkan ini (dari dataset kita)
    'img_size_w': 64,    # Tambahkan ini (dari dataset kita)
    'patch_size': PATCH_SIZE,
    'num_classes': NUM_CLASSES,
    'd_model': D_MODEL,
    'n_head': N_HEAD,
    'n_layers': N_LAYERS,
    'ffn_hidden': FFN_HIDDEN,
    'drop_prob': DROP_PROB,
    'device': device
}

# Inisialisasi model
model = AMCTransformer(**model_params).to(device)

# Inisialisasi Loss dan Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# Variabel untuk Early Stopping
best_val_loss = float('inf')
patience_counter = 0
best_model_weights = None

# Riwayat untuk plotting
history = {
    'train_loss': [],
    'val_loss': [],
    'val_acc': []
}

print(f"✅ Model AMCTransformer diinisialisasi dengan {sum(p.numel() for p in model.parameters()):,} parameter.")

Menginisialisasi model...
✅ Model AMCTransformer diinisialisasi dengan 399,493 parameter.


In [ ]:
print("🚀 Memulai Training...")

for epoch in range(NUM_EPOCHS):
    print(f"\n--- Epoch {epoch+1}/{NUM_EPOCHS} ---")
    
    # --- FASE TRAINING ---
    model.train()
    running_train_loss = 0.0
    
    # Gunakan tqdm untuk progress bar
    train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1} [Train]", unit="batch")
    
    for images, labels, _ in train_pbar:
        images, labels = images.to(device), labels.to(device)
        
        # Nol-kan gradien
        optimizer.zero_grad()
        
        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # Backward pass dan optimasi
        loss.backward()
        optimizer.step()
        
        running_train_loss += loss.item()
        train_pbar.set_postfix(loss=f"{loss.item():.4f}")
    
    epoch_train_loss = running_train_loss / len(train_loader)
    history['train_loss'].append(epoch_train_loss)
    print(f"Epoch {epoch+1} Selesai. Rata-rata Train Loss: {epoch_train_loss:.4f}")
    
    # --- FASE VALIDASI ---
    model.eval()
    running_val_loss = 0.0
    correct = 0
    total = 0
    
    val_pbar = tqdm(valid_loader, desc=f"Epoch {epoch+1} [Valid]", unit="batch")
    
    with torch.no_grad():
        for images, labels, _ in val_pbar:
            images, labels = images.to(device), labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_val_loss += loss.item()
            
            # Hitung akurasi
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
            val_pbar.set_postfix(loss=f"{loss.item():.4f}")

    epoch_val_loss = running_val_loss / len(valid_loader)
    epoch_val_acc = 100 * correct / total
    history['val_loss'].append(epoch_val_loss)
    history['val_acc'].append(epoch_val_acc)
    
    print(f"Rata-rata Valid Loss: {epoch_val_loss:.4f} | Valid Acc: {epoch_val_acc:.2f}%")
    
    # --- Cek Early Stopping ---
    if epoch_val_loss < best_val_loss:
        print(f"💡 Valid Loss membaik: {best_val_loss:.4f} -> {epoch_val_loss:.4f}. Menyimpan model...")
        best_val_loss = epoch_val_loss
        patience_counter = 0
        best_model_weights = copy.deepcopy(model.state_dict())
    else:
        patience_counter += 1
        print(f"⌛ Valid Loss tidak membaik. Kesabaran: {patience_counter}/{patience}")

    if patience_counter >= patience:
        print(f"🛑 Early stopping di epoch {epoch+1}.")
        break

print("\n🏁 Training Selesai.")
# Muat bobot model terbaik
if best_model_weights:
    print("Memuat bobot model terbaik untuk evaluasi...")
    model.load_state_dict(best_model_weights)

🚀 Memulai Training...

--- Epoch 1/100 ---


Epoch 1 [Train]:   0%|          | 0/5822 [00:00<?, ?batch/s]

Epoch 1 Selesai. Rata-rata Train Loss: 0.9608


Epoch 1 [Valid]:   0%|          | 0/1666 [00:00<?, ?batch/s]

Rata-rata Valid Loss: 0.8318 | Valid Acc: 58.48%
💡 Valid Loss membaik: inf -> 0.8318. Menyimpan model...

--- Epoch 2/100 ---


...

Epoch 28 [Train]:   0%|          | 0/5822 [00:00<?, ?batch/s]

Epoch 28 Selesai. Rata-rata Train Loss: 0.7139


Epoch 28 [Valid]:   0%|          | 0/1666 [00:00<?, ?batch/s]

Rata-rata Valid Loss: 0.7160 | Valid Acc: 63.80%
💡 Valid Loss membaik: 0.7184 -> 0.7160. Menyimpan model...

--- Epoch 29/100 ---


Epoch 29 [Train]:   0%|          | 0/5822 [00:00<?, ?batch/s]

In [ ]:
import os

# Create result directory if it doesn't exist
os.makedirs('result', exist_ok=True)

print("--- 1. Plotting Kurva Pembelajaran ---")
plt.figure(figsize=(12, 4))
# Plot Loss
plt.subplot(1, 2, 1)
plt.plot(history['train_loss'], label='Train Loss')
plt.plot(history['val_loss'], label='Valid Loss')
plt.title('Kurva Loss Training vs Validasi')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
# Plot Akurasi
plt.subplot(1, 2, 2)
plt.plot(history['val_acc'], label='Valid Accuracy', color='green')
plt.title('Kurva Akurasi Validasi')
plt.xlabel('Epoch')
plt.ylabel('Akurasi (%)')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('result/learning_curves.png', dpi=600, bbox_inches='tight')
plt.show()


print("\n--- 2. Evaluasi pada Test Set ---")
model.eval()
all_labels = []
all_predictions = []
all_snrs = []

test_pbar = tqdm(test_loader, desc="Mengevaluasi Test Set", unit="batch")

with torch.no_grad():
    for images, labels, snrs in test_pbar:
        images, labels = images.to(device), labels.to(device)
        
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        
        all_labels.extend(labels.cpu().numpy())
        all_predictions.extend(predicted.cpu().numpy())
        all_snrs.extend(snrs.cpu().numpy())

# Konversi ke array NumPy
y_true = np.array(all_labels)
y_pred = np.array(all_predictions)
snrs_list = np.array(all_snrs)

# Hitung akurasi keseluruhan
overall_accuracy = accuracy_score(y_true, y_pred)
print(f"Akurasi Keseluruhan pada Test Set: {overall_accuracy * 100:.2f}%")


print("\n--- 3. Plotting Confusion Matrix ---")
# Buat label string dari label map
labels_str = list(label_map.keys())

cm = confusion_matrix(y_true, y_pred)
# Normalisasi CM
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

plt.figure(figsize=(10, 8))
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues', 
            xticklabels=labels_str, yticklabels=labels_str)
plt.title(f'Confusion Matrix (Akurasi: {overall_accuracy*100:.2f}%)')
plt.ylabel('Label Sebenarnya')
plt.xlabel('Label Prediksi')
plt.savefig('result/confusion_matrix.png', dpi=600, bbox_inches='tight')
plt.show()


print("\n--- 4. Plotting Akurasi vs. SNR ---")
snr_values = np.unique(snrs_list)
acc_per_snr = []

for snr in snr_values:
    # Dapatkan indeks untuk SNR spesifik ini
    idx = (snrs_list == snr)
    if np.sum(idx) > 0: # Pastikan ada sampel
        snr_acc = accuracy_score(y_true[idx], y_pred[idx])
        acc_per_snr.append(snr_acc)
    else:
        acc_per_snr.append(np.nan) # Tambah NaN jika tidak ada sampel

plt.figure(figsize=(10, 6))
plt.plot(snr_values, [a * 100 for a in acc_per_snr], 'o-', label='Akurasi Model')
plt.title('Akurasi vs. SNR pada Test Set')
plt.xlabel('SNR (dB)')
plt.ylabel('Akurasi (%)')
plt.grid(True)
plt.xticks(snr_values[::2]) # Tampilkan label SNR setiap 4dB
plt.ylim(0, 100) # Akurasi dari 0% sampai 100%
plt.legend()
plt.savefig('result/accuracy_vs_snr.png', dpi=600, bbox_inches='tight')
plt.show()

# Terakhir, tutup file HDF5
print("Membersihkan dataset...")
train_dataset.close()
valid_dataset.close()
test_dataset.close()
print("✅ Selesai.")
print(f"📁 Semua plot telah disimpan di direktori 'result/' dengan resolusi 600 DPI")

In [ ]:
# --- Confusion Matrix Visualization ---
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# Buat label string dari label_map
labels_str = list(label_map.keys())

# Hitung confusion matrix
cm = confusion_matrix(y_true, y_pred)

# Normalisasi per baris (true label)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

# Plot confusion matrix
plt.figure(figsize=(10, 8))
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=labels_str, yticklabels=labels_str)
plt.title(f'Confusion Matrix (Akurasi: {overall_accuracy*100:.2f}%)')
plt.ylabel('Label Sebenarnya')
plt.xlabel('Label Prediksi')

# Simpan hasil ke folder result
os.makedirs('result', exist_ok=True)
plt.savefig('result/confusion_matrix.png', dpi=600, bbox_inches='tight')
plt.show()